In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# ==============================================================================
# PHASE 1: DATA INGESTION & NORMALIZATION
# ==============================================================================
def normalize_zillow_matrix(df, value_column_name):
    """
    Transforms Zillow's wide-format economic matrices into a normalized
    time-series format optimized for BI ingestion and financial modeling.
    """

    # Strip redundant geographic granularity to save memory.
    # Zillow flags everything here as 'msa' (Metropolitan Statistical Area).
    if 'RegionType' in df.columns:
        df = df.drop(columns=['RegionType'])

    metadata_cols = ['RegionID', 'SizeRank', 'RegionName', 'StateName']

    # Unpivot the historical timeline. Zillow stores dates as columns (wide format);
    # we need a proper time-series index (long format) for the arbitrage calculations.
    df_long = pd.melt(df, id_vars=metadata_cols, var_name='Date', value_name=value_column_name)

    # Purge incomplete market histories and the aggregated 'National' baseline.
    # We only want actionable, metro-level data.
    df_long = df_long.dropna(subset=[value_column_name, 'StateName'])

    # Defensive check: Ensure no overlapping historical records for a single market
    df_long = df_long.drop_duplicates(subset=['RegionID', 'Date'])

    # Enforce standard datetime typing for strict time-series sorting later
    df_long['Date'] = pd.to_datetime(df_long['Date'])

    return df_long

In [3]:
# ==============================================================================
# PIPELINE EXECUTION
# ==============================================================================

print("[ETL] Ingesting raw Zillow Home Value (ZHVI) and Rental (ZORI) indices...")
#zhvi stands for Zillow Home Value Index. Represents the "typical" home value (35th-65th percentile), removing luxury outliers.
#zori stands for Zillow Observed Rent Index	Tracks the "market-rate" rent for repeat-units, ensuring a like-for-like comparison over time.
zhvi_path = '/content/drive/MyDrive/project_portfolio/housing/metro_zhvi.csv'
zori_path = '/content/drive/MyDrive/project_portfolio/housing/metro_zori.csv'

df_homes_raw = pd.read_csv(zhvi_path)
df_rentals_raw = pd.read_csv(zori_path)

print(f"[ETL] Raw files loaded. ZHVI shape: {df_homes_raw.shape} | ZORI shape: {df_rentals_raw.shape}")
print("[ETL] Normalizing matrices into time-series format...")

df_homes_clean = normalize_zillow_matrix(df_homes_raw, 'Home_Value')
df_rentals_clean = normalize_zillow_matrix(df_rentals_raw, 'Rent_Value')

print("[ETL] Merging property values and rental rates...")
# Lock the dataframes together using the unique Zillow RegionID and Date.
# Inner join ensures we only evaluate markets where we have BOTH home prices and rent data.
final_df = pd.merge(
    df_homes_clean[['RegionID', 'SizeRank', 'RegionName', 'StateName', 'Date', 'Home_Value']],
    df_rentals_clean[['RegionID', 'Date', 'Rent_Value']],
    on=['RegionID', 'Date'],
    how='inner'
)

# Post-merge optimization: Drop the internal Zillow ID as it's no longer needed for BI mapping
final_df = final_df.drop(columns=['RegionID'])

print("[ETL] Pushing clean pipeline to storage...")
export_path = '/content/drive/MyDrive/project_portfolio/housing/zillow_arbitrage_clean.csv'
final_df.to_csv(export_path, index=False)

print("-" * 50)
print("SUCCESS: Master ETL Pipeline Complete.")
print(f"Final shape pushed to BI: {final_df.shape}")
print("-" * 50)

[ETL] Ingesting raw Zillow Home Value (ZHVI) and Rental (ZORI) indices...
[ETL] Raw files loaded. ZHVI shape: (895, 318) | ZORI shape: (721, 138)
[ETL] Normalizing matrices into time-series format...
[ETL] Merging property values and rental rates...
[ETL] Pushing clean pipeline to storage...
--------------------------------------------------
SUCCESS: Master ETL Pipeline Complete.
Final shape pushed to BI: (47846, 6)
--------------------------------------------------
